# MCTS -> Policy MLP Trainer

This notebook clones/opens `showdown-trainer`, runs the sanity checks, then trains a small masked policy MLP from JSONL MCTS examples. Checkpoints include both `model_state_dict` and `optimizer_state_dict`, so you can resume training by pointing `RESUME_FROM` at an existing checkpoint.

In [ ]:
# Clone/setup cell. If this notebook is already inside the repo, it uses the current folder.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/YOUR_USERNAME/showdown-trainer.git"  # <-- edit this
BRANCH = None  # e.g. "main" or None

here = Path.cwd()
if (here / "train.py").exists() and (here / "dataset.py").exists():
    repo_dir = here
else:
    repo_dir = here / "showdown-trainer"
    if not (repo_dir / "train.py").exists():
        clone_cmd = ["git", "clone"]
        if BRANCH:
            clone_cmd += ["--branch", BRANCH]
        clone_cmd += [REPO_URL, str(repo_dir)]
        subprocess.run(clone_cmd, check=True)

os.chdir(repo_dir)
print(f"Working in: {Path.cwd()}")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
# Test run: checks mapping, serialization, encoding, dataset writing, checkpointing, and training.
import subprocess
import sys

subprocess.run([sys.executable, "sanity_check.py"], check=True)


In [ ]:
# Configure a training batch. Replace DATA_PATH with real MCTS JSONL once collected.
from pathlib import Path
from dataset import TrainingDatasetWriter

DATA_PATH = Path("training_data/mcts_run.jsonl")
CHECKPOINT_PATH = Path("checkpoints/policy_mlp.pt")
RESUME_FROM = CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else None

# Optional: create a tiny demo dataset so the notebook can run before you collect real battles.
CREATE_DEMO_IF_MISSING = True

if CREATE_DEMO_IF_MISSING and not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    writer = TrainingDatasetWriter(DATA_PATH, run_id="notebook-demo")
    base_state = {
        "battle_tag": "battle-gen9ou-demo",
        "pokemon_format": "gen9ou",
        "generation": "gen9",
        "turn": 1,
        "user": {
            "active": {
                "name": "greattusk",
                "moves": [{"name": "earthquake"}, {"name": "protect"}],
                "can_terastallize": True,
            },
            "reserve": [{"name": "dragapult", "hp": 100}],
        },
        "opponent": {"active": {"name": "kingambit"}, "reserve": []},
    }
    demo_policies = [
        {"earthquake": 0.8, "protect": 0.1, "switch dragapult": 0.1},
        {"earthquake": 0.2, "protect": 0.7, "switch dragapult": 0.1},
        {"earthquake": 0.4, "protect": 0.2, "switch dragapult": 0.4},
        {"earthquake": 0.6, "protect": 0.2, "switch dragapult": 0.2},
        {"earthquake": 0.1, "protect": 0.8, "switch dragapult": 0.1},
        {"earthquake": 0.3, "protect": 0.3, "switch dragapult": 0.4},
    ]
    for index, policy in enumerate(demo_policies, start=1):
        state = dict(base_state)
        state["turn"] = index
        chosen_action = max(policy, key=policy.get)
        record = writer.write_decision(
            state=state,
            chosen_action=chosen_action,
            mcts_policy=policy,
        )
        writer.write_result(battle_id=record.battle_id, winner="bot")

print(f"DATA_PATH={DATA_PATH.resolve()}")
print(f"CHECKPOINT_PATH={CHECKPOINT_PATH.resolve()}")
print(f"RESUME_FROM={RESUME_FROM}")


In [ ]:
# Training batch. Re-run this cell to continue from CHECKPOINT_PATH after the first checkpoint exists.
from train import TrainConfig, train

config = TrainConfig(
    data_path=str(DATA_PATH),
    output_path=str(CHECKPOINT_PATH),
    resume_checkpoint_path=str(RESUME_FROM) if RESUME_FROM else None,
    epochs=2,                 # additional epochs to run in this batch
    batch_size=2,
    validation_split=0.33,
    hidden_sizes=(64,),       # bump to (1024, 512) for real runs
    dropout=0.0,
    hash_buckets=256,         # bump to 4096 for real runs
    device="auto",
)

summary = train(config)
RESUME_FROM = CHECKPOINT_PATH
summary


In [ ]:
# Inspect checkpoint and run one masked prediction from a saved example.
from dataset import iter_decision_records
from encode import EncoderConfig
from train import load_policy_checkpoint, predict_policy

model, checkpoint = load_policy_checkpoint(CHECKPOINT_PATH)
assert "model_state_dict" in checkpoint
assert "optimizer_state_dict" in checkpoint

first_record = next(iter_decision_records(DATA_PATH))
encoder_config = EncoderConfig(**checkpoint["encoder_config"])
probs = predict_policy(model, first_record["state"], encoder_config=encoder_config)

print("checkpoint epoch:", checkpoint["epoch"])
print("model keys:", list(checkpoint["model"].keys()))
print("optimizer state groups:", len(checkpoint["optimizer_state_dict"].get("param_groups", [])))
print("policy probs:", [round(p, 4) for p in probs])
print("sum:", round(sum(probs), 6))
